Bot behavior from security gateways is primarily click-based. The gateway scans the email by following links, which generates the suspicious timing patterns. An opens-only subscriber could be a real person who reads but never clicks, or it could be a bot, but have no timing signal to distinguish them from opens alone. Can't detect bots without click data.
Therefore, opens-only subscribers are not detectable with current feature set. They stay in the dataset as unclassified, neither flagged as bot nor confirmed human.

In [ ]:
# Import libararies
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Read dataset
df = pd.read_csv("activity_log_raw.csv")
print(df.info())
print(df.shape)
df.head()

A total of 127,715 activity logs were pulled on the 1,000 sample subscribers.

In [ ]:
# Converting date columns to datetime
df['created_at'] = pd.to_datetime(df['created_at'])
df['updated_at'] = pd.to_datetime(df['updated_at'])

df.info()

In [ ]:
# verifying what subject id column is
df['subject_id'].duplicated().sum()

In [ ]:
# View all types of values for subject_type column
print(df['subject_type'].unique())
print(df['log_name'].unique())

The `log_name` and `subject_type` columns seem to be related with how related each columns' values are. Suspecting the `log_name` is the subcategories of `subject_type`. The filter below confirms which event types fall under the `subject_type`.

In [ ]:
# Check log_name values under the email subject_type
df[df['subject_type'] == 'email']['log_name'].unique()

# All event type combinations and their counts
df[['subject_type', 'log_name']].value_counts()

**subject_type** -- the category of object the event relates to
- clicks -- link click events
- opens -- email open events  
- campaigns -- campaign-level events (sends)
- email -- automation and bounce events
- subscribers -- account-level events (signup, group changes, etc.)

**log_name** -- the specific event type within each subject_type category

**Keep:** clicks and opens only, all others dropped, not relevant to bot detection

In [ ]:
# Checking cteated_at and updated_at columns
print(df[df['subscriber_id'] == 170176106616325149]) # subscriber_id retrieved from activity_log_fetch.ipynb
(df['created_at'] == df['updated_at']).all() # are all of them identical?

In [ ]:
# Inspect a specific subject_id
df[df['subject_id'] == 187409537911752556]

`id` is the whole event. Every row has its own unique id. It identifies the event itself, not what the event is about.

`subject_id` behaves differently by event type. For *campaign_send* events (`log_name`), `subject_id` equals *campaign_id* and *automation_id* (nested dict inside `properties`), it points to the original email. For *email_open* and *link_click* events, `subject_id` equals `id`, it self-references. MailerLite appears to give opens and clicks the same value for `id` and `subject_id` to distinguish them from email-session-level activity.

*campaign_id* and *automation_id* inside `properties` is the universal email identifier. It is consistent across all event types regardless of how `subject_id` behaves. For opens and clicks, it is the only place the actual email id lives.
- Both are foreign keys to email sends and serve as the session grouping key when combined with subscriber_id to correctly isolate each subscriber's opens and clicks within a single email send for feature engineering.


| Column | Definition | Decision | Reason |
|--------|------------|----------|--------|
| `id` | Primary key of the activity log table, uniquely identifies each event row | Drop | No grouping power. Feature engineering aggregates to campaign level, not row level |
| `log_name` | The activity being done at a granular level (email_open, link_click, etc.) | Drop | Redundant with subject_type after filtering to opens and clicks only |
| `subject_id` | Behaves as a foreign key for campaign_send events (points to campaign). Self-references id for opens and clicks | Drop | Not a reliable foreign key to campaign for opens and clicks. campaign_id from properties used instead |
| `subject_type` | Category for all subcategories in log_name (clicks, opens, campaigns, etc.) | Keep | Used for row filtering and retained to distinguish opens from clicks |
| `properties` | Nested dict with event metadata, ex. campaign_id, campaign_name, preview_url, etc. | Extract then drop | Contains campaign_id and automation_id, the true foreign key to the emails. Extract before dropping |
| `created_at` | Timestamp of when the event occurred, primary feature engineering input | Keep | All timing features will be computed from this column |
| `updated_at` | Timestamp of last record update, identical to created_at in all observed records | Drop | Redundant with created_at |
| `subscriber_id` | Foreign key to the active subscribers dataset | Keep | Half of the composite session grouping key (subscriber_id + campaign_id) |
| `campaign_id` | Extracted from propertiesm, the foreign key to the campaigns | Keep | Half of the composite session grouping key. Universal across all event types |

In [ ]:
# Dictionary inside properties converted to strings when saved as CSV
# Convert it back to a dict first using ast
df['properties'] = df['properties'].apply(ast.literal_eval)

# Extracting the `campaign_id` inside the properties column
df['campaign_id'] = df['properties'].apply(lambda x: x.get('campaign_id') if isinstance(x, dict) else None)
df['automation_id'] = df['properties'].apply(lambda x: x.get('automation_id') if isinstance(x,dict) else None)

df.info()
df.head()

Rows from other event types (campaign sends, bounces, subscriber events) return NaN for both `campaign_id` and `automation_id`. Only `opens` and `clicks` rows are kept for feature engineering.

In [ ]:
print(f"Row counts should be {df['subject_type'].isin(['opens', 'clicks']).sum()} after filter.\n")

# Dropping rows
open_click_df = df[df['subject_type'].isin(['opens', 'clicks'])].copy()
print(open_click_df.shape)
print(open_click_df['subject_type'].value_counts())

print()
print(open_click_df['subscriber_id'].nunique())

`campaign_id` and `automation_id` are extracted from `properties` to identify which email each event belongs to. Automation emails are checked separately. 

In [ ]:
# Checking the NAs in campaign_id and automation_id
print(open_click_df['campaign_id'].notna().sum())
print(open_click_df['automation_id'].notna().sum())
print((open_click_df['campaign_id'] == (open_click_df['automation_id'])).sum()) # to see if any campaign ids and automation ids are the same

In [ ]:
# Checking the automation emails
print(open_click_df['automation_id'].nunique())
print(open_click_df.groupby(['subscriber_id', 'automation_id']).size())

In [ ]:
# Get subscribers recieving the automation emails
automation_subs = open_click_df[open_click_df['automation_id'].notna()]['subscriber_id'].unique()

# Check if they also appear in campaign emails
sub_auto_camp = open_click_df[open_click_df['subscriber_id'].isin(automation_subs) & open_click_df['campaign_id'].notna()]['subscriber_id'].nunique()
print(f"Of the {len(automation_subs)} subscribers recieving automation emails, {sub_auto_camp} are also receiving campaign emails.")

Automation emails are excluded from feature engineering. The same `automation_id` is sent repeatedly to the same subscribers over time. These values will corrupt timing features by artifically inflating the features. Most subscribers receiving automation emails also have campaign activity and are retained through that data. `automation_id` is dropped along with the other columns identified in the table above.

In [ ]:
# Dropping columns
cleaned_df = open_click_df.drop(columns=[
    'id',
    'log_name',
    'subject_id',
    'properties',
    'updated_at',
    'automation_id'
])
print(cleaned_df.shape)
cleaned_df.head()

After `automation_id` was dropped, some of the values for `campaign_id` remain NaNs after the rows and columns remove step above. These NaN values will be dropped as well.

In [ ]:
# Filter out the NaN in campaign id
cleaned_df = cleaned_df[cleaned_df['campaign_id'].notna()]
print(cleaned_df.shape)

# Varify total number of subscribers
print(cleaned_df['subscriber_id'].nunique())

In [ ]:
# Renaming the created_at to an intuitive name
cleaned_df = cleaned_df.rename(columns={
    'created_at' : 'event_time',
    'campaign_id' : 'email_id'})
cleaned_df.head()

Feature engineering is email-campaign-level, not event-level (row-level). Bot detection features are computed by grouping events into email sessions defined by `subscriber_id` + `email_id`. 

clicks before open:
- When any click event in a session has a timestamp earlier than the earliest open event in that same session, or when clicks exist but no open event exists at all
- Not humanly possible to have any click before opening the email, as a human cannot click a link in an email they haven't opened yet.

open to first click latency:
- Seconds gap between the first open event and the first click event in the same session
- Instantaneous click and open are not realistic human behaviors

min time between clicks:
- Seconds gap between any two consecutive click events in the same session
- Instantaneous clicking multiple times is not realistic

clicks per session:
- Total distinct click events recorded within a single session window
- The ratio of clicks to an email opened can signal bot activity. Ex. A security gateway clicks every link in the email to scan for threats, while a human clicks one or two links they're interested in.

After these four features are computed, they are aggregated to subscriber level before modeling. Clicks before open is aggregated as a proportion of flagged sessions over total sessions. The remaining three features are aggregated as averages, representing each subscriber's typical latency and clicking behavior across all sessions rather than any single campaign.

In [ ]:
# Step 1 - Filter to clicks-only df and a opens-only df
clicks = cleaned_df[cleaned_df['subject_type'] == 'clicks']
opens = cleaned_df[cleaned_df['subject_type'] == 'opens']

# First click event per session
first_click = clicks.groupby(['subscriber_id', 'email_id'])['event_time'].min().rename('first_clk_time')
display(first_click)

# First open event per session
first_open = opens.groupby(['subscriber_id', 'email_id'])['event_time'].min().rename('first_opn_time')
display(first_open)

In [ ]:
# Step 2 - concat both df 
session_df = pd.concat([first_click, first_open], axis=1)

# Step 3 - compute the difference 
session_df['clks_b4_opn'] = (
    (session_df['first_clk_time'] < session_df['first_opn_time']) | # checks click before open
    (session_df['first_clk_time'].notna() & session_df['first_opn_time'].isna()) # Checks for clicks but no opn
    ).astype(int) # change Tures and Falses into 1|0
print((session_df['clks_b4_opn'] == 1).value_counts())
session_df

Sessions where a click occured before the open, or where clicks occurred with no open recorded, are flagged as `clks_b4_opn = 1`.

`first_clk_time` and `first_opn_time` are in datetime dtype. Subtracting them produces a timedelta, so need to convert to seconds to get a numeric latency value for each session. Sessions with negative latency are already captured by `clks_b4_opn` and are set to NaN to avoid diluting the near-zero latency signal.


In [ ]:
# Compute latency between email opened and first click
session_df['opn_to_first_clk_gap'] = (session_df['first_clk_time'] - session_df['first_opn_time']).dt.total_seconds() # converting from timedelta64 to seconds after diff
print(session_df['opn_to_first_clk_gap'].value_counts())
print()
print(session_df.info())
session_df

There are negative values in the `opn_to_first_clk_gap` column. These values resulted from emails sessions with clicks occuring before the email open. Since these are already engineered as the `clks_b4_opn`, they will be replaced with NaN

In [ ]:
# Replacing negative values with NaN
session_df.loc[session_df['opn_to_first_clk_gap'] < 0, 'opn_to_first_clk_gap'] = np.nan
print(session_df['opn_to_first_clk_gap'].value_counts())
print()
print(session_df.info())

Each row in `session_df` represents one session, defined as a unique subscriber and email combination.
| Column | Description |
|--------|------------|
|`first_clk_time`| Contains only the first click of the email, used to engineer features|
|`first_opn_time`| Contains only the first open of the email, used to engineer features|
|`clks_b4_opn`| Feature. 1 represent either clicks before open or clicks without any open, 0 represents emails were opened first or has no clicks|
|`opn_to_first_clk_gap`| Feature. The latency between email opened and first link clicked|

The next feature to compute is the minimum time between two clicks. In order to compute the difference between two consecutive clicks, the timestamps need to be in chronological order.

In [ ]:
# Computing for the minimum time gap between two consecutive clicks in each email

# Use the clicks df from Step 1 above to sort the event_time
click_sorted = clicks.sort_values(['subscriber_id', 'email_id', 'event_time'])

# Calculate the difference for the click timestamps within each email session, then convert to seconds
time_diffs = click_sorted.groupby(['subscriber_id', 'email_id'])['event_time'].diff().dt.total_seconds()

# Add the subscriber_id and email_id back to the df after performing .diff()
# Take only the minimum time per email session
time_diffs = time_diffs.groupby([click_sorted['subscriber_id'], click_sorted['email_id']]).min().rename('min_clk_time_gap')
time_diffs

In [ ]:
# Inspect the df
print(time_diffs.info())
display(time_diffs.describe())

In [ ]:
# Checking the NaNs in the min_clk_time_gap
print(time_diffs[time_diffs.isna()])

# Step 1 count all clicks per email 
clicks_per_email = click_sorted.groupby(['subscriber_id', 'email_id']).size().rename('click_counts') # this is also one of the four features, will be merged

# Step 2 merge the time_diffs to clicks_per_email
check_merge_df = pd.concat([time_diffs, clicks_per_email], axis=1)

# Step 3 check the nans in min_clk_time_gap
check_merge_df[check_merge_df['min_clk_time_gap'].isna()]

NaN values in `min_clk_time_gap` represent sessions with only one click event. The *.diff()* function performed on the email group level, requires at least two values to compute. Hence, emails with one click resulted in NaN values. These NaNs won't be dropped because they still have information with other features after it's merged with the session_df. 

In [ ]:
# Merge the time_diff and session_df and clicks_per_email
session_df_2 = pd.concat([session_df, check_merge_df], axis=1).reset_index().copy()
print(session_df_2.info())
session_df_2

Due to different shapes between `time_diff` and `session_df` and `clicks_per_email`, concatenation fills empty cells from `min_clk_time_gap` and `click_counts` with NaN values. These are sessions with opens but no clicks have no corresponding rows in either `min_clk_time_gap` or `clicks_per_email` df. \
`min_clik_time_gap` end up with *two NaN sources*: NaN from .diff() computation and from concat shape mismatch.

Data manipulation can introduce errors. Need to verify each subscriber and respective email contains the correct features.

In [ ]:
# Aggregate the features to subscriber level
sub_act_agg = session_df_2.groupby('subscriber_id').agg(
    clks_b4_opn_prop = ('clks_b4_opn', 'mean'), # operationally, it's simply the average because it's only 1's and 0's
    opn_to_first_clk_gap_avg = ('opn_to_first_clk_gap', 'mean'),
    min_clk_sec_gap_avg = ('min_clk_time_gap', 'mean'),
    click_counts_avg = ('click_counts', 'mean')
).reset_index()
print(sub_act_agg.info())
display(sub_act_agg.describe())
sub_act_agg.head()

In [ ]:
# 97th percentile used to evaluate cap threshold for min_clk_sec_gap_avg NaN replacement
sub_act_agg['min_clk_sec_gap_avg'].quantile(.97)

The sample was designed with three engagement tiers, so the NaN distribution across features is expected rather than a data quality issue. Subscribers with no clicks have no values for `click_counts_avg`, and those with no campaign clicks have no values for `opn_to_first_clk_gap_avg`. `min_clk_sec_gap_avg` has two NaN sources: sessions with only one click from `.diff()`, and zero-click subscribers from the concat shape mismatch.

| Features | Bot Interpretation | Total NaN | NaN Interpretation | NaN Decision|
|-----|------|------|------|------|
| `clks_b4_opn_prop` | higher = more clicks before opens = stronger bot signal | 0 | N/A | N/A |
| `opn_to_first_clk_gap_avg` | lower = sooner clicks after open = stronger bot signal | 329 | subscribers had no clicks or had clicks before opens | replacing with median is better because mean is inflated by extreme outliers|
| `min_clk_sec_gap_avg` | lower = faster consecutive clicks = stronger bot signal | 563 | subscribers had only single-click sessions or zero clicks across all sessions | upon confirming the NaNs with `click_counts_avg`, all NaN values will be replaced with the value at 99th percentile because they are more likely to be human behaviors |
| `click_counts_avg` | higher = more clicks per session = stronger bot signal | 323 | subscribers had no click events at all | replacing with 0 because they simply have no clicks|

In [ ]:
# Confirming if all click_counts_avg NaN values overlap with NaNs of min_clk_sec_gap_avg
sub_act_agg[sub_act_agg['click_counts_avg'].isna() & 
            sub_act_agg['min_clk_sec_gap_avg'].isna()].describe()

In [ ]:
# Validating the NaNs for opn_to_first_clk_gap_avg
display(sub_act_agg[sub_act_agg['opn_to_first_clk_gap_avg'].isna() & 
                    sub_act_agg['click_counts_avg'].isna()].describe()) # Result: 323 NaNs overlapping, CONFIRMED

# Validating if the 6 remaining NaNs in opn_to_first_clk_gap_avg overlapping with clks_b4_opn_prop
display(sub_act_agg[sub_act_agg['opn_to_first_clk_gap_avg'].isna() & 
                    sub_act_agg['click_counts_avg'].notna()])

sub_act_agg[sub_act_agg['opn_to_first_clk_gap_avg'].isna() & 
            sub_act_agg['click_counts_avg'].notna()]['clks_b4_opn_prop'] # Result: 6 NaNs overlapping, CONFIRMED

 NaN overlap across all three features confirms the expected structure. The `click_counts_avg` NaNs are fully contained within `min_clk_sec_gap_avg` NaNs. The remaining `min_clk_sec_gap_avg` NaNs are single-click subscribers. The remaining `opn_to_first_clk_gap_avg` NaNs beyond the zero-click overlap are clicks-before-open subscribers confirmed by non-zero `clks_b4_opn_prop`. NaN replacement decisions in the table above remain unchanged.

In [ ]:
# Replacing NaN values 
sub_act_agg['opn_to_first_clk_gap_avg'] = sub_act_agg['opn_to_first_clk_gap_avg'].fillna(sub_act_agg['opn_to_first_clk_gap_avg'].median())
sub_act_agg['min_clk_sec_gap_avg'] = sub_act_agg['min_clk_sec_gap_avg'].fillna(sub_act_agg['min_clk_sec_gap_avg'].quantile(.99))
sub_act_agg['click_counts_avg'] = sub_act_agg['click_counts_avg'].fillna(0)
print(sub_act_agg.info())
display(sub_act_agg.describe())
sub_act_agg.head()

The means and medians dropped across the board for those three columns, which is expected. Adding more values compressed the spread between percentiles as well. \
It is worth noting that the 50th and 75th percentile for `min_clk_sec_gap_avg` are the same. 

In [ ]:
#  Aggregated subscriber-level features are saved to CSV
sub_act_agg.to_csv('subscriber_activity_aggregated.csv', index=False)